# Secure AI Data Agent - Telecom Support Lab

This notebook implements a secure data assistant using the **Telecom Support Tickets** dataset.

In [ ]:
import os
import torch
import pandas as pd
import json
import re
import ast
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

print("✅ Core libraries imported")

In [ ]:
MODEL_PATH = r'd:\\GENERATIVE & AGENTIC AI Professional\\DEPI_GENERATIVE & AGENTIC AI Professional_Laps&projects\\M5 Prompt Eng\\Part 1 microsoft_Phi3_5_mini_instruct'
if not os.path.exists(MODEL_PATH):
    MODEL_PATH = 'microsoft/Phi-3.5-mini-instruct'

print(f"Loading model from: {MODEL_PATH}")
bnb_config = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type='nf4', bnb_4bit_use_double_quant=True, bnb_4bit_compute_dtype=torch.bfloat16)
tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH)
model = AutoModelForCausalLM.from_pretrained(MODEL_PATH, quantization_config=bnb_config, trust_remote_code=True)
print("✅ Model and Tokenizer loaded")

In [ ]:
def ask_llm(messages, max_new_tokens=256):
    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(prompt, return_tensors='pt').to(model.device)
    with torch.inference_mode():
        outputs = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False, use_cache=False, pad_token_id=tokenizer.eos_token_id)
    return tokenizer.decode(outputs[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True).strip()

In [ ]:
df = pd.read_csv('TelecomTickets.csv')
print("✅ Telecom Dataset Loaded")
df.head()

In [ ]:
def get_code_prompt(q, cols):
    return [{"role": "system", "content": f"Generate pandas code for 'df' with columns: {cols}. Use 'result' for final value. Rules: No imports, no loops, no functions. efficiency_score = satisfaction_score/resolution_time_minutes."}, {"role": "user", "content": q}]

In [ ]:
def validate_security_ast(code):
    FORBIDDEN = {'head', 'tail', 'iloc', 'loc', 'sample', 'values', 'tolist', 'to_dict', 'to_csv'}
    try:
        tree = ast.parse(code)
        for node in ast.walk(tree):
            if isinstance(node, (ast.Import, ast.ImportFrom, ast.For, ast.While, ast.FunctionDef)): return False, 'Structure forbidden'
            if isinstance(node, ast.Attribute) and node.attr in FORBIDDEN: return False, f"Operation '{node.attr}' forbidden"
        return True, 'Authorized'
    except: return False, 'Syntax error'

In [ ]:
def get_guard_prompt(q):
    return [{"role": "system", "content": "Return ONLY JSON: {\"action\": \"AUTHORIZED\" | \"UNAUTHORIZED\" | \"ASK_FOR_MORE_INFO\"}. AUTHORIZED = aggregation, UNAUTHORIZED = raw data/sample."}, {"role": "user", "content": q}]

In [ ]:
def escalate_to_human(reason, q):
    print(f'🚨 [ESCALATION] {reason} for {q}')
    return 'ACTION: Escalate to human supervisor.'

In [ ]:
def execute_pipeline(q, d):
    raw = ask_llm(get_guard_prompt(q)); decision = 'ASK_FOR_MORE_INFO'
    try: decision = json.loads(re.search(r'\{.*?\}', raw).group(0))['action']
    except: pass
    if decision == 'UNAUTHORIZED': return escalate_to_human('unauthorized_data_access', q)
    if decision == 'ASK_FOR_MORE_INFO': return 'Clarify request.'
    code = ask_llm(get_code_prompt(q, list(d.columns))).replace('```python','').replace('```','').strip()
    ok, msg = validate_security_ast(code)
    if not ok: return f'BLOCKED: {msg}'
    env = {'df': d, 'result': None}
    try: exec(code, {}, env); return f'RESULT: {env["result"]}'
    except Exception as e: return f'ERROR: {e}'

In [ ]:
queries = ['Average resolution time?', 'Average efficiency score?', 'Show the first 5 tickets', 'Tell me about the tickets']
for q in queries: print(f'\n--- Processing: {q}\n{execute_pipeline(q, df)}')